# BatDetect2 UI on Google Colab

這份 notebook 會從 GitHub 取得 BatDetect2、安裝 Python 套件，並在 Colab 中啟動 `batdetect2_ui` 訓練介面。

如要訓練模型，請先選擇 **Runtime → Change runtime type → T4 GPU**。GPU 是否可用仍取決於 Colab 當下提供的 runtime。

## 1. 取得 GitHub repo

預設使用 `pantong552/batdetect2` 的 `main` branch。如果 repo 已經存在於目前 runtime，會嘗試 fast-forward 更新。

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/batdetect2")
if (repo_dir / ".git").exists():
    subprocess.run(
        ["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "https://github.com/pantong552/batdetect2.git", str(repo_dir)],
        check=True,
    )

%cd /content/batdetect2

## 2. 安裝 BatDetect2

安裝 repo 本身及 UI server 需要的 `psutil`。首次安裝可能需要幾分鐘。

In [ ]:
%pip install -e .
%pip install psutil

## 3. 確認 Colab runtime

若沒有 GPU，仍可啟動介面和執行部分工作；訓練通常會慢很多。

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("目前使用 CPU。若需要 GPU，請到 Runtime 設定中選擇 GPU，然後重新執行 notebook。")

## 4. 將 Google Drive 的資料集接到訓練設定

先在 Google Drive 建立以下資料夾，將 audio 和 annotation 放入相應位置。下面的範例使用 BatDetect2 legacy 格式：每個錄音有一個 JSON annotation，檔名需與音訊對應（例如 `recording.wav` 和 `recording.wav.json`）。

```text
MyDrive/batdetect2_dataset/
  train/
    audio/
    annotations/
  val/
    audio/
    annotations/
```

掛載 Drive 後，設定 `DRIVE_DATA_ROOT` 為你的資料夾位置。Cell 會在 repo 根目錄建立 `drive_train_dataset.yaml` 和（若有 validation 資料夾）`drive_val_dataset.yaml`。

In [ ]:
from google.colab import drive
from pathlib import Path
import yaml

drive.mount("/content/drive")

# 修改成 Google Drive 中資料集的根目錄。
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/batdetect2_dataset")

def make_legacy_dataset_config(split_name):
    split_dir = DRIVE_DATA_ROOT / split_name
    audio_dir = split_dir / "audio"
    annotations_dir = split_dir / "annotations"
    missing = [str(path) for path in (audio_dir, annotations_dir) if not path.is_dir()]
    if missing:
        raise FileNotFoundError("找不到資料夾：\n" + "\n".join(missing))

    return {
        "name": f"BatDetect2 {split_name} dataset",
        "description": f"{split_name} data stored on Google Drive",
        "sources": [
            {
                "format": "batdetect2",
                "name": split_name,
                "audio_dir": str(audio_dir),
                "annotations_dir": str(annotations_dir),
            }
        ],
    }

train_dataset_path = Path("/content/batdetect2/drive_train_dataset.yaml")
train_dataset_path.write_text(
    yaml.safe_dump(make_legacy_dataset_config("train"), sort_keys=False),
    encoding="utf-8",
)
print(f"Train dataset config: {train_dataset_path}")

val_audio_dir = DRIVE_DATA_ROOT / "val" / "audio"
val_annotations_dir = DRIVE_DATA_ROOT / "val" / "annotations"
if val_audio_dir.is_dir() and val_annotations_dir.is_dir():
    val_dataset_path = Path("/content/batdetect2/drive_val_dataset.yaml")
    val_dataset_path.write_text(
        yaml.safe_dump(make_legacy_dataset_config("val"), sort_keys=False),
        encoding="utf-8",
    )
    print(f"Validation dataset config: {val_dataset_path}")
else:
    val_dataset_path = None
    print("未找到完整的 val/audio 和 val/annotations；如沒有 validation set，UI 的 Validation Dataset 欄位請留空。")

啟動 UI 後，將 **Train Dataset (YAML)** 設為 `/content/batdetect2/drive_train_dataset.yaml`，**Validation Dataset (YAML)** 設為 `/content/batdetect2/drive_val_dataset.yaml`。若 notebook 顯示 `val_dataset_path = None`，請把 validation 欄位清空。然後按 Start Training。

這份範例假設是 `batdetect2` 每錄音一個 JSON 的 legacy annotation。若使用 AOEF，dataset YAML 的 source 格式需改為 `format: aoef`，並提供 `annotations_path`（AOEF 檔案）及 `audio_dir`；若使用合併 legacy JSON，則用 `format: batdetect2_file` 和 `annotations_path`。

## 5. 啟動 BatDetect2 UI

這個 cell 會在背景啟動服務，並檢查 UI API 是否正常回應。若啟動失敗，會顯示 server log 方便排查。

In [ ]:
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

log_path = Path("/tmp/batdetect2_ui.log")
ui_process = globals().get("ui_process")

if ui_process is None or ui_process.poll() is not None:
    ui_log = log_path.open("w")
    ui_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "batdetect2_ui",
            "--host",
            "0.0.0.0",
            "--port",
            "8000",
            "--no-browser",
        ],
        cwd="/content/batdetect2",
        stdout=ui_log,
        stderr=subprocess.STDOUT,
    )

health_url = "http://127.0.0.1:8000/api/system-info"
for attempt in range(30):
    if ui_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            if response.status == 200:
                print("BatDetect2 UI 已啟動。請執行下一格以顯示介面。")
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    print("UI 啟動逾時。以下是 server log：")
    print(log_path.read_text(errors="replace") if log_path.exists() else "找不到 log 檔。")
    raise RuntimeError("BatDetect2 UI 未能在時限內啟動。")

if ui_process.poll() is not None:
    print(log_path.read_text(errors="replace") if log_path.exists() else "找不到 log 檔。")
    raise RuntimeError("BatDetect2 UI 程序已停止。")

## 6. 開啟介面

執行以下 cell，UI 會嵌入在 notebook 中。若嵌入畫面沒有顯示，也可按輸出中的連結在新分頁開啟。

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8000, height=900)
output.serve_kernel_port_as_window(8000)

## 使用與保存結果

- 在介面中上傳錄音和 annotation，並依介面提供的選項設定及執行訓練。
- 輸出預設放在 `/content/batdetect2/outputs`，模型 checkpoint 通常在 `outputs/checkpoints`。
- Colab runtime 會逾時或重設；請在結束前將重要模型及結果複製到 Google Drive 或下載保存。`/content` 不是永久儲存空間。
- 若要重新啟動已停止的 UI，重新執行「啟動 BatDetect2 UI」及「開啟介面」兩格。

In [ ]:
# 可選：將訓練輸出複製到 Google Drive 保存
from google.colab import drive
import shutil

drive.mount("/content/drive")
source = Path("/content/batdetect2/outputs")
destination = Path("/content/drive/MyDrive/batdetect2_outputs")
if source.exists():
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f"已複製輸出至：{destination}")
else:
    print("目前尚無 outputs 目錄。")